Crear Embeddings y Medir Similitud

In [1]:
!pip install sentence-transformers

from sentence_transformers import SentenceTransformer, util

modelo = SentenceTransformer('all-MiniLM-L6-v2')

texto1 = "Fallo de autenticación en SSH"
texto2 = "Intento de login fallido en el puerto 22"
texto3 = "Compra de boletos de avión"

emb1 = modelo.encode(texto1, convert_to_tensor=True)
emb2 = modelo.encode(texto2, convert_to_tensor=True)
emb3 = modelo.encode(texto3, convert_to_tensor=True)

# Convertimos a valores flotantes en lugar de tensores
sim_12 = util.cos_sim(emb1, emb2).item()
sim_13 = util.cos_sim(emb1, emb3).item()

print("🔹 Similitud entre texto 1 y texto 2 (deben ser similares):")
print(f"{sim_12:.4f}")

print("\n🔹 Similitud entre texto 1 y texto 3 (deben ser distintos):")
print(f"{sim_13:.4f}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

🔹 Similitud entre texto 1 y texto 2 (deben ser similares):
0.5348

🔹 Similitud entre texto 1 y texto 3 (deben ser distintos):
0.2343


Mini Transformer para entender "self-attention"

In [ ]:
import torch
import torch.nn.functional as F

tokens = ["servidor", "puerto", "22", "expuesto"]
vocab = {w:i for i,w in enumerate(tokens)}
one_hot = torch.eye(len(tokens))

query = one_hot[vocab["puerto"]]
keys = one_hot

attention_scores = torch.matmul(keys, query)
attention_probs = F.softmax(attention_scores, dim=0)

print("Tokens:", tokens)
print("Attention hacia 'puerto':")
for t, p in zip(tokens, attention_probs):
    print(f"{t}: {p:.4f}")


Tokens: ['servidor', 'puerto', '22', 'expuesto']
Attention hacia 'puerto':
servidor: 0.1749
puerto: 0.4754
22: 0.1749
expuesto: 0.1749


Detección rápida de anomalías con IA

In [ ]:
!pip install sentence-transformers
from sentence_transformers import SentenceTransformer, util
import numpy as np

# Cargamos el modelo
modelo = SentenceTransformer('all-MiniLM-L6-v2')

logs = [
    "Failed password for admin from 192.168.1.10 port 22",
    "Failed password for admin from 192.168.1.11 port 22",
    "Failed password for admin from 192.168.1.12 port 22",
    "Successful login for admin from 203.0.113.55 port 22", # ← raro
]

emb = modelo.encode(logs, convert_to_tensor=True)

sim = util.cos_sim(emb, emb)
import numpy as np

distances = 1 - sim.mean(dim=1).cpu().numpy()

print("≠ Mayor distancia → Más anomalía")
for log, score in zip(logs, distances):
    print(f"{score:.4f} → {log}")


≠ Mayor distancia → Más anomalía
0.0819 → Failed password for admin from 192.168.1.10 port 22
0.0808 → Failed password for admin from 192.168.1.11 port 22
0.0901 → Failed password for admin from 192.168.1.12 port 22
0.2023 → Successful login for admin from 203.0.113.55 port 22


Hallucination test (demostración de limitación de LLM)

In [ ]:
!pip install transformers accelerate

from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="gpt2",
    device=0   # CPU
)

prompt = "Explain the technical details of the cyber vulnerability CVE-2099-9999 in IPv9 quantum networks."

output = generator(
    prompt,
    max_length=150,
    temperature=1.5,
    top_k=20,
    repetition_penalty=1.0,
    do_sample=True
)

print("❗ Prompt:", prompt)
print("➡ Respuesta del modelo (alucinación):")
print(output[0]["generated_text"])




/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Device set to use cuda:0
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


❗ Prompt: Explain the technical details of the cyber vulnerability CVE-2099-9999 in IPv9 quantum networks.
➡ Respuesta del modelo (alucinación):
Explain the technical details of the cyber vulnerability CVE-2099-9999 in IPv9 quantum networks. [CVE-2013-3030] (CVE-2013-3034) WebAssembly 2.7 and WebKit 2.7 as they appear (CVE-2013-3037) when WebKit is installed on mobile devices. [CVE-2013-3028] Microsoft Windows Phone 10, as specified by Microsoft, WebKit in Apple Safari 2 and Android apps on an embedded Intel Xeon Phi Processor. [CVE-2013-3027] Windows 10 10 and WebKit 2.1.1 [CERT-2013-01-2045, CVE-2013-3020], as they appear on embedded platforms, does not prevent denial of service and could allow vectors such as crafted HTML, Javascript and JavaScript in the web browser to execute arbitrary code. [CVE-2013-3020] Windows 7 10 Pro, the minimum required upgrade for Microsoft Windows 10.1, does not work correctly on mobile devices. [CVE-2013-3011] V8.2.0.1101 and earlier may cause an infor

Prueba con SeeD establecido

In [ ]:
from transformers import pipeline
import torch

# Fijar semilla aleatoria para obtener resultados consistentes
torch.manual_seed(42)  # Establecer semilla de PyTorch para consistencia

# Cargar el modelo GPT-2
llm = pipeline("text-generation", model="gpt2", device=0)  # Cambiado a -1 para CPU

# Función para generar una respuesta simple en inglés (sin razonamiento)
def generate_simple_response(prompt):
    # Generar respuesta simple, concisa y directa en inglés
    return llm(prompt, max_length=50, truncation=True, pad_token_id=50256)[0]["generated_text"]

# Pregunta en inglés
prompt = "What is an LLM?"

# Generación de respuesta simple en inglés
simple_response = generate_simple_response(prompt)

# Mostrar la respuesta simple en inglés
print("Simple response in English:")
print(simple_response)

Device set to use cuda:0
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Simple response in English:
What is an LLM?

There aren't lots of LLM implementations. LLM is a combination of common libraries that combine the best of all possible approaches, and which are designed to be used in a single application. LLM is a high-level abstraction layer used to make applications more modular, and enables the use of a wide range of different LLM features and libraries.

There are different ways in which a library can be used in a single project. For example, if you have many LLM implementations, you can use them to write your own application. However, this is not the same as using the same library for a single application.

Why is LLM so useful?

It is a high-level abstraction layer that enables the use of many different LLM features and libraries. It can be used to write a simple application without having to write large classes or large applications. In order to use LLM, you need to use a wide range of libraries, and you need to read and write your own code in man

Prueba con Seed pero con Promt de Razonamiento

In [ ]:
from transformers import pipeline
import torch

# Fijar semilla aleatoria para consistencia
torch.manual_seed(42)

# Cargar el modelo GPT-2
llm = pipeline("text-generation", model="gpt2", device=0)  # Cambiar a 0 si tienes GPU, sino usa -1 para CPU

# Función para "thinking" (con razonamiento)
def generate_with_thinking(prompt):
    # Crear una cadena de razonamiento, añadiendo explicaciones intermedias.
    chain_of_thought = f"Time to thinking about Generative AI \n\n {prompt} "

    # Generar la respuesta con el razonamiento
    response = llm(chain_of_thought, max_length=200, truncation=True, pad_token_id=50256)
    return response[0]["generated_text"]

# Pregunta
prompt = "What is an LLMs in the context of Artificial Intelligent?"

# Generación de respuesta con razonamiento
thinking_response = generate_with_thinking(prompt)

# Mostrar la respuesta
print("Response with Thinking (Chain of Thought):")
print(thinking_response)


Device set to use cuda:0
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Response with Thinking (Chain of Thought):
Time to thinking about Generative AI 

 What is an LLMs in the context of Artificial Intelligent? 

I am interested in the term "AI", but in general it is defined as human-based AI. In the field of Artificial Intelligence, for example, the concepts are like this:


Intelligent machines are computers that can do anything they want (like inventing things, inventing things. They are intelligent). The more intelligent the machine, the more intelligent the machine will be. And this is true for humans.


Intelligent machines are machines that can execute anything they want (like inventing things, inventing things. They are intelligent). The more intelligent the machine, the more intelligent the machine will be. And this is true for humans. A machine that has a good sense of humor, a clever mind, or a great imagination can make an intelligent decision about a problem. Or it can be a machine that does a great job at solving a problem.


Intelligent ma

Combinaciones de Temperatura... Mas temperatura mas "creativo"

In [ ]:
from transformers import pipeline
import torch, random, numpy as np

# ---- util: semilla global (compatible con tu pipeline)
def set_seed_all(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    # (Opcional) mayor determinismo en CUDA; puede bajar performance
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# 1) Carga pipeline
pipe = pipeline("text-generation",
                model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
                device_map="auto")

# 2) Prompt en formato chat
prompt = pipe.tokenizer.apply_chat_template(
    [
        {"role": "system", "content": "Eres docente"},
        {"role": "user", "content": "Define 'embedding' en una sola frase."}
    ],
    tokenize=False,
    add_generation_prompt=True
)

EOS = pipe.model.config.eos_token_id
PAD = pipe.model.config.eos_token_id

# 3) Línea base determinista (greedy)
out = pipe(prompt, max_new_tokens=100, do_sample=False,
           return_full_text=False, eos_token_id=EOS, pad_token_id=PAD)
print("Greedy (sin sampling) ->", out[0]["generated_text"].strip())

# 4) Comparación de temperatures (reproducible con semilla global)
for t, seed in [(0.2, 42), (0.7, 42), (1.2, 42)]:
    set_seed_all(seed)  # misma "suerte" para cada temperatura
    out = pipe(prompt,
               max_new_tokens=100,
               do_sample=True,
               temperature=t,
               top_p=0.9,
               return_full_text=False,
               eos_token_id=EOS, pad_token_id=PAD)
    print(f"Temp {t} ->", out[0]["generated_text"].strip())

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Greedy (sin sampling) -> En una sola frase, 'embedding' significa la incorporación de una representación de texto en una red neuronal artificial (RNA) para mejorar su capacidad de entendimiento y aprendizaje. La representación de texto se almacena en la memoria de la RNA y se utiliza para entrenar la red neuronal. La RNA luego se utiliza para entrenar una red neuronal más avanzada que puede
Temp 0.2 -> En una sola frase, 'embedding' significa la incorporación de una o más palabras o frases en otro texto o material, generalmente para proporcionar información o contexto. En el contexto de la lingüística, 'embedding' se utiliza para describir la forma en que las palabras se incorporan a otros textos o materiales, como en el caso de la traducción de textos de una lengua a otra.
Temp 0.7 -> En una sola frase, se refiere a la representación de una palabra o frase en un espacio o sistema de representación que es idéntico a la representación de esa palabra en el espacio de entornos o contextos